# PostgreSQL Database Setup for UFA Data

This notebook:
1. Creates the database schema (games, teams, players, events, line_players)
2. Provides functions to insert cleaned event data into PostgreSQL
3. Filters events to only store relevant types

## Setup and Configuration

In [2]:
# Install required packages (run once)
# %pip install psycopg2-binary

In [3]:
import psycopg2
from psycopg2.extras import execute_batch
from typing import List, Dict, Optional
import json

In [4]:
# Database connection configuration
DB_CONFIG = {
    'dbname': 'ufa_analytics',
    'user': os.getenv('DB_USER', 'postgres'),
    'password': '',             # ← No password for local setup
    'host': 'localhost',
    'port': 5432
}

def get_connection():
    """Create and return a database connection."""
    return psycopg2.connect(**DB_CONFIG)

## Event Type Filter

Only these event types will be stored in the database:

In [ ]:
# Event types to keep in database
KEEP_EVENT_TYPES = {
    # Core gameplay
    1,   # Start O Point
    2,   # Start D Point
    7,   # Pull - inbounds
    8,   # Pull - out of bounds
    11,  # Block
    16,  # Foul (called by)
    17,  # Foul (called on)
    18,  # Pass
    19,  # Goal
    20,  # Drop
    22,  # Throwaway
    23,  # Callahan
    24,  # Stall
    
    # Optional tracking
    3,   # Midpoint Timeout
    21,  # Dropped Pull
    25,  # Injury
    26,  # Player Misconduct
    27,  # Player Ejected
    
    # Period markers
    28, 29, 30, 31, 32, 33  # End Q1, Q2, Q3, Q4, OT1, OT2
}

# Event types that have line data (need entries in line_players table)
LINE_EVENT_TYPES = {1, 2, 3, 25}

def should_keep_event(event_type: int) -> bool:
    """Check if an event type should be stored in the database."""
    return event_type in KEEP_EVENT_TYPES

## Database Schema Creation

In [6]:
def create_schema():
    """Create all database tables."""
    conn = get_connection()
    cur = conn.cursor()
    
    # Drop existing tables (careful!)
    cur.execute("""
        DROP TABLE IF EXISTS line_players CASCADE;
        DROP TABLE IF EXISTS events CASCADE;
        DROP TABLE IF EXISTS games CASCADE;
        DROP TABLE IF EXISTS players CASCADE;
        DROP TABLE IF EXISTS teams CASCADE;
    """)
    
    # Create games table
    cur.execute("""
        CREATE TABLE games (
            game_id VARCHAR(50) PRIMARY KEY,
            home_team_id VARCHAR(50) NOT NULL,
            away_team_id VARCHAR(50) NOT NULL,
            home_score INT,
            away_score INT,
            game_date DATE,
            year INT NOT NULL,
            created_at TIMESTAMP DEFAULT NOW()
        );
    """)
    
    # Create teams table
    cur.execute("""
        CREATE TABLE teams (
            team_id VARCHAR(50) PRIMARY KEY,
            team_name VARCHAR(100),
            division VARCHAR(50),
            created_at TIMESTAMP DEFAULT NOW()
        );
    """)
    
    # Create players table
    cur.execute("""
        CREATE TABLE players (
            player_id VARCHAR(50) PRIMARY KEY,
            full_name VARCHAR(100),
            team_id VARCHAR(50),
            year INT,
            created_at TIMESTAMP DEFAULT NOW()
        );
    """)
    
    # Create events table
    cur.execute("""
        CREATE TABLE events (
            event_id SERIAL PRIMARY KEY,
            game_id VARCHAR(50) NOT NULL REFERENCES games(game_id),
            event_number INT NOT NULL,
            event_type INT NOT NULL,
            team VARCHAR(50),
            year INT NOT NULL,
            
            -- Time field (for types 1, 2)
            time INT,
            
            -- Pull fields (for types 7, 8, 9, 10)
            puller VARCHAR(50),
            pull_x DOUBLE PRECISION,
            pull_y DOUBLE PRECISION,
            pull_ms INT,
            
            -- Throw/Pass fields (for types 18, 19, 20, 21)
            thrower VARCHAR(50),
            thrower_x DOUBLE PRECISION,
            thrower_y DOUBLE PRECISION,
            receiver VARCHAR(50),
            receiver_x DOUBLE PRECISION,
            receiver_y DOUBLE PRECISION,
            
            -- Turnover location fields (for types 22, 23)
            turnover_x DOUBLE PRECISION,
            turnover_y DOUBLE PRECISION,
            
            -- Defender field (for types 11, 12)
            defender VARCHAR(50),
            
            -- Player field (for types 26, 27)
            player VARCHAR(50),
            
            -- Custom fields
            synthetic BOOLEAN DEFAULT FALSE,
            
            created_at TIMESTAMP DEFAULT NOW(),
            
            CONSTRAINT unique_event_position UNIQUE(game_id, event_number)
        );
    """)
    
    # Create indexes on events
    cur.execute("""
        CREATE INDEX idx_events_game_id ON events(game_id);
        CREATE INDEX idx_events_type ON events(event_type);
        CREATE INDEX idx_events_team ON events(team);
        CREATE INDEX idx_events_synthetic ON events(synthetic);
        CREATE INDEX idx_events_thrower ON events(thrower);
        CREATE INDEX idx_events_receiver ON events(receiver);
        CREATE INDEX idx_events_defender ON events(defender);
    """)
    
    # Create line_players table
    cur.execute("""
        CREATE TABLE line_players (
            line_player_id SERIAL PRIMARY KEY,
            event_id INT NOT NULL REFERENCES events(event_id) ON DELETE CASCADE,
            game_id VARCHAR(50) NOT NULL,
            player_id VARCHAR(50) NOT NULL,
            line_type CHAR(1),
            position_order INT,
            team VARCHAR(50) NOT NULL,
            created_at TIMESTAMP DEFAULT NOW(),
            
            CONSTRAINT unique_player_per_event UNIQUE(event_id, player_id)
        );
    """)
    
    # Create indexes on line_players
    cur.execute("""
        CREATE INDEX idx_line_players_event ON line_players(event_id);
        CREATE INDEX idx_line_players_player ON line_players(player_id);
        CREATE INDEX idx_line_players_game ON line_players(game_id);
        CREATE INDEX idx_line_players_team ON line_players(team);
        CREATE INDEX idx_line_players_type ON line_players(line_type);
    """)
    
    conn.commit()
    cur.close()
    conn.close()
    
    print("✅ Database schema created successfully!")

In [ ]:
# Create the schema (COMMENTED OUT - only run once manually, not on every ingestion)
# create_schema()

✅ Database schema created successfully!


## Data Insertion Functions

In [8]:
def insert_game(game_id: str, home_team_id: str, away_team_id: str, 
                home_score: int, away_score: int, year: int):
    """Insert a game record."""
    conn = get_connection()
    cur = conn.cursor()
    
    # Extract date from game_id (format: YYYY-MM-DD-TEAM-TEAM)
    game_date = game_id[:10]  # '2024-04-27'
    
    cur.execute("""
        INSERT INTO games (game_id, home_team_id, away_team_id, home_score, away_score, game_date, year)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT (game_id) DO NOTHING;
    """, (game_id, home_team_id, away_team_id, home_score, away_score, game_date, year))
    
    conn.commit()
    cur.close()
    conn.close()
    print(f"✅ Inserted game: {game_id}")

In [ ]:
def insert_events(game_id: str, events: List[Dict]):
    """
    Insert events into the database.
    Filters out event types we don't want to keep.
    Returns a mapping of event_number to database event_id for line_players insertion.
    """
    conn = get_connection()
    cur = conn.cursor()
    
    event_id_map = {}  # Maps event_number to database event_id
    events_to_insert = []
    
    for idx, event in enumerate(events, 1):
        event_type = event.get('type')
        
        # Skip events we don't want to keep
        if not should_keep_event(event_type):
            continue
        
        # Build event data
        event_data = (
            game_id,
            idx,  # event_number
            event_type,
            event.get('team'),
            event.get('year'),
            event.get('time'),
            event.get('puller'),
            event.get('pullX'),
            event.get('pullY'),
            event.get('pullMs'),
            event.get('thrower'),
            event.get('throwerX'),
            event.get('throwerY'),
            event.get('receiver'),
            event.get('receiverX'),
            event.get('receiverY'),
            event.get('turnoverX'),
            event.get('turnoverY'),
            event.get('defender'),
            event.get('player'),
            event.get('synthetic', False)
        )
        events_to_insert.append(event_data)
    
    # Batch insert events (ON CONFLICT DO NOTHING for re-runs)
    execute_batch(cur, """
        INSERT INTO events (
            game_id, event_number, event_type, team, year,
            time, puller, pull_x, pull_y, pull_ms,
            thrower, thrower_x, thrower_y, receiver, receiver_x, receiver_y,
            turnover_x, turnover_y, defender, player,
            synthetic
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT (game_id, event_number) DO NOTHING;
    """, events_to_insert)
    
    # Query back all event_ids for this game (avoids execute_batch RETURNING chunk bug)
    cur.execute("SELECT event_number, event_id FROM events WHERE game_id = %s", (game_id,))
    for event_number, event_id in cur.fetchall():
        event_id_map[event_number] = event_id
    
    conn.commit()
    cur.close()
    conn.close()
    
    print(f"✅ Inserted {len(events_to_insert)} events for game {game_id}")
    return event_id_map

In [ ]:
def insert_line_players(game_id: str, events: List[Dict], event_id_map: Dict[int, int]):
    """
    Insert line player data for events that have line information.
    
    Note: In the UFA data format, event_type=1 is logged by the PULLING (defensive) team,
    so their line is the D-line. event_type=2 is logged by the RECEIVING (offensive) team,
    so their line is the O-line.
    """
    conn = get_connection()
    cur = conn.cursor()
    
    line_data = []
    
    for idx, event in enumerate(events, 1):
        event_type = event.get('type')
        
        # Only process line events that were inserted
        if event_type not in LINE_EVENT_TYPES or idx not in event_id_map:
            continue
        
        event_id = event_id_map[idx]
        line = event.get('line', [])
        team = event.get('team')
        
        # Determine line type (O or D)
        # event_type=1: pulling team's D-line; event_type=2: receiving team's O-line
        line_type = 'D' if event_type == 1 else 'O' if event_type == 2 else None
        
        # Insert each player in the line
        for position, player_id in enumerate(line, 1):
            line_data.append((
                event_id,
                game_id,
                player_id,
                line_type,
                position,
                team
            ))
    
    if line_data:
        execute_batch(cur, """
            INSERT INTO line_players (event_id, game_id, player_id, line_type, position_order, team)
            VALUES (%s, %s, %s, %s, %s, %s)
            ON CONFLICT (event_id, player_id) DO NOTHING;
        """, line_data)
    
    conn.commit()
    cur.close()
    conn.close()
    
    print(f"✅ Inserted {len(line_data)} line player records for game {game_id}")

In [11]:
def insert_players(players: List[Dict]):
    """
    Insert players into the database.
    
    Args:
        players: List of player dictionaries from get_players_data()
                 Expected fields: player_id, full_name, team_id, year
    """
    conn = get_connection()
    cur = conn.cursor()
    
    players_data = []
    for player in players:
        players_data.append((
            player['player_id'],
            player['full_name'],
            player['team_id'],
            player['year']
        ))
    
    if players_data:
        execute_batch(cur, """
            INSERT INTO players (player_id, full_name, team_id, year)
            VALUES (%s, %s, %s, %s)
            ON CONFLICT (player_id) DO UPDATE
            SET full_name = EXCLUDED.full_name,
                team_id = EXCLUDED.team_id,
                year = EXCLUDED.year;
        """, players_data)
    
    conn.commit()
    cur.close()
    conn.close()
    
    print(f"✅ Inserted/updated {len(players_data)} players")

In [12]:
def insert_teams(teams: List[Dict]):
    """
    Insert teams into the database.
    
    Args:
        teams: List of team dictionaries from get_teams_data()
               Expected fields: team_id, team_name, division, year
    """
    conn = get_connection()
    cur = conn.cursor()
    
    teams_data = []
    for team in teams:
        teams_data.append((
            team['team_id'],
            team['team_name'],
            team['division']
        ))
    
    if teams_data:
        execute_batch(cur, """
            INSERT INTO teams (team_id, team_name, division)
            VALUES (%s, %s, %s)
            ON CONFLICT (team_id) DO UPDATE
            SET team_name = EXCLUDED.team_name,
                division = EXCLUDED.division;
        """, teams_data)
    
    conn.commit()
    cur.close()
    conn.close()
    
    print(f"✅ Inserted/updated {len(teams_data)} teams")

## Teams and Players Insertion

## Complete Insertion Pipeline

In [13]:
def insert_game_data(game_id: str, events: List[Dict], 
                     home_team_id: str, away_team_id: str,
                     home_score: int, away_score: int, year: int):
    """
    Complete pipeline to insert all data for a game.
    
    Args:
        game_id: Game identifier (e.g., '2024-04-27-MTL-NY')
        events: List of event dictionaries from clean_data + insert_missing_turnovers
        home_team_id: Home team ID
        away_team_id: Away team ID
        home_score: Final home team score
        away_score: Final away team score
        year: Game year
    """
    print(f"\n{'='*60}")
    print(f"Inserting data for game: {game_id}")
    print(f"{'='*60}")
    
    # 1. Insert game record
    insert_game(game_id, home_team_id, away_team_id, home_score, away_score, year)
    
    # 2. Insert events (returns mapping of event_number to event_id)
    event_id_map = insert_events(game_id, events)
    
    # 3. Insert line players
    insert_line_players(game_id, events, event_id_map)
    
    print(f"\n✅ Complete! Game {game_id} inserted successfully.")
    print(f"{'='*60}\n")

## Example Usage

This shows how to use the insertion functions with data from `data_cleaning_functions.ipynb`

In [14]:
# Example: Insert a game
# First, you would run data_cleaning_functions.ipynb to get cleaned events:

# from data_cleaning_functions import clean_data, insert_missing_turnovers
# 
# game_ids = ["2024-04-27-MTL-NY"]
# cleaned_data = clean_data(game_ids)
# 
# for game_id, events in cleaned_data.items():
#     # Apply missing turnovers fix
#     fixed_events = insert_missing_turnovers(events, seed=42)
#     
#     # Calculate final score by counting goals
#     home_score = sum(1 for e in fixed_events if e.get('type') == 19 and e.get('team') == 'royal')
#     away_score = sum(1 for e in fixed_events if e.get('type') == 19 and e.get('team') == 'empire')
#     
#     # Insert into database
#     insert_game_data(
#         game_id=game_id,
#         events=fixed_events,
#         home_team_id='royal',
#         away_team_id='empire',
#         home_score=home_score,
#         away_score=away_score,
#         year=2024
#     )

## Verification Queries

In [15]:
def verify_insertion(game_id: str):
    """Run verification queries to check data was inserted correctly."""
    conn = get_connection()
    cur = conn.cursor()
    
    print(f"\nVerifying data for game: {game_id}\n")
    
    # Count events by type
    cur.execute("""
        SELECT event_type, COUNT(*) as count
        FROM events
        WHERE game_id = %s
        GROUP BY event_type
        ORDER BY event_type;
    """, (game_id,))
    
    print("Events by type:")
    for row in cur.fetchall():
        print(f"  Type {row[0]}: {row[1]} events")
    
    # Count synthetic events
    cur.execute("""
        SELECT COUNT(*)
        FROM events
        WHERE game_id = %s AND synthetic = TRUE;
    """, (game_id,))
    synthetic_count = cur.fetchone()[0]
    print(f"\nSynthetic events: {synthetic_count}")
    
    # Count line players
    cur.execute("""
        SELECT COUNT(*)
        FROM line_players
        WHERE game_id = %s;
    """, (game_id,))
    line_count = cur.fetchone()[0]
    print(f"Line player records: {line_count}")
    
    # Show final score
    cur.execute("""
        SELECT home_team_id, home_score, away_team_id, away_score
        FROM games
        WHERE game_id = %s;
    """, (game_id,))
    game_info = cur.fetchone()
    if game_info:
        print(f"\nFinal Score: {game_info[0]} {game_info[1]} - {game_info[2]} {game_info[3]}")
    
    cur.close()
    conn.close()

In [16]:
# Verify the insertion
# verify_insertion("2024-04-27-MTL-NY")